# 05 – Iteration 3: Model Training (Random Forest)
This notebook trains a RandomForest model on the sampled, preprocessed dataset.
It loads the preprocessed train/valid/test sets, fits the model, evaluates it,
and saves both the model and the evaluation metrics.


In [1]:
import os, json
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, classification_report
import joblib

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
PREPARED_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'results', 'prepared'))
MODELS_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'results', 'models'))
os.makedirs(MODELS_DIR, exist_ok=True)

# Load preprocessed data
X_train = np.load(os.path.join(PREPARED_DIR, 'X_train.npy'))
X_valid = np.load(os.path.join(PREPARED_DIR, 'X_valid.npy'))
X_test  = np.load(os.path.join(PREPARED_DIR, 'X_test.npy'))

y_train = np.load(os.path.join(PREPARED_DIR, 'y_train.npy'))
y_valid = np.load(os.path.join(PREPARED_DIR, 'y_valid.npy'))
y_test  = np.load(os.path.join(PREPARED_DIR, 'y_test.npy'))

print('Loaded shapes:')
print('X_train:', X_train.shape)
print('X_valid:', X_valid.shape)
print('X_test :', X_test.shape)

rf = RandomForestClassifier(
    n_estimators=200,
    n_jobs=-1,
    random_state=42,
    class_weight='balanced_subsample'
)

rf.fit(X_train, y_train)
print('RandomForest trained.')

def evaluate(clf, X, y):
    y_proba = clf.predict_proba(X)[:,1]
    y_pred = clf.predict(X)
    try:
        roc = roc_auc_score(y, y_proba)
    except:
        roc = None
    try:
        pr = average_precision_score(y, y_proba)
    except:
        pr = None
    cm = confusion_matrix(y, y_pred).tolist()
    rep = classification_report(y, y_pred, output_dict=True, zero_division=0)
    return {'roc_auc': roc, 'pr_auc': pr, 'confusion_matrix': cm, 'classification_report': rep}

metrics_valid = evaluate(rf, X_valid, y_valid)
metrics_test  = evaluate(rf, X_test,  y_test)

# Save model and metrics
joblib.dump(rf, os.path.join(MODELS_DIR, 'model_iter3_rf.joblib'))
with open(os.path.join(MODELS_DIR, 'metrics_iter3_rf.json'), 'w') as f:
    json.dump({'valid': metrics_valid, 'test': metrics_test}, f, indent=2)

print('Saved model and metrics.')


Loaded shapes:
X_train: (740570, 7)
X_valid: (159100, 7)
X_test : (160127, 7)
RandomForest trained.


d:\joan\CONDA_ENVS\ab_datachallenge\lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
d:\joan\CONDA_ENVS\ab_datachallenge\lib\site-packages\sklearn\metrics\_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
d:\joan\CONDA_ENVS\ab_datachallenge\lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
d:\joan\CONDA_ENVS\ab_datachallenge\lib\site-packages\sklearn\metrics\_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


Saved model and metrics.
